# 114. Flatten Binary Tree to Linked List

**Difficulty:** Medium &nbsp;|&nbsp; **Topics:** tree, dfs, stack, linked list
&nbsp;|&nbsp; [LeetCode](https://leetcode.com/problems/flatten-binary-tree-to-linked-list/)

Given the `root` of a binary tree, **flatten it into a "linked list" in-place**.

- The "linked list" uses the same `TreeNode` class, where the `right` child
  pointer points to the next node and the `left` child pointer is always `null`.
- The linked list should be in the same order as a **pre-order traversal** of the
  binary tree.

---

### Example 1

```
Input:  root = [1,2,5,3,4,null,6]
Output: [1,null,2,null,3,null,4,null,5,null,6]
```

```
        1                    1
       / \                    \
      2   5        ->           2
     / \   \                     \
    3   4   6                      3
                                    \
                                     4
                                      \
                                       5
                                        \
                                         6
```

### Example 2

```
Input:  root = []
Output: []
```

### Example 3

```
Input:  root = [0]
Output: [0]
```

---

### Constraints

- The number of nodes in the tree is in the range `[0, 2000]`.
- `-100 <= Node.val <= 100`

### Follow up

Can you flatten the tree **in-place** with `O(1)` extra space?

## Before you write anything

Answer these on paper. No code yet.

**1.** Look at Example 1's output: `1, 2, 3, 4, 5, 6`. Now look at the input tree.
What traversal produces that exact order? (You already wrote this traversal in
the Org Chart TP - `printPersons`.)

**2.** So one obvious plan is: *collect the nodes in that order into a list, then
re-link them.* Write down what "re-link" means precisely - for two consecutive
nodes `a` and `b` in your list, what **two** assignments do you make?

**3.** Now the trap. Suppose you try to do it directly at node `1` and your first
move is:

```
root.right = root.left
```

Draw the tree and ask: **where is the subtree rooted at 5 now?** Can you still
reach it? What should you have done first?

That is the same bug family as Reverse Linked List (#206) - you overwrote a
pointer before saving what it pointed to.

**4.** In the finished list, node `2` is followed by `3`, and node `4` is followed
by `5`. But `4` and `5` are in *different subtrees* - `4` is the deepest-right
node of `1`'s **left** subtree, and `5` is `1`'s **right** child.

So: after the whole left subtree is flattened into a chain, **which node in that
chain must point to the old right subtree?** Describe that node in words
("the ... node of ...").

If you can answer 4, you have the `O(1)`-space solution and you do not need a
list or a stack at all.

## Three routes - do at least two

**A - list + re-link** *(the one your answers to 1 & 2 describe)*
Pre-order walk collecting nodes into a Python list, then a second pass to wire
them up. `O(n)` time, `O(n)` space. Easiest to get right first.

**B - explicit stack** *(this is why the problem is in your stack list)*
Push the root. Pop a node, and push its children - **in which order**, so that
popping gives you pre-order? A stack is LIFO, so if you want `left` out first,
which one goes in first? Wire each popped node to the next one you pop.
`O(n)` time, `O(h)` space.

**C - in-place, `O(1)` space** *(the follow-up, from your answer to 4)*
Walk down the `right` spine with one pointer. At each node that has a left child:
find the node you named in question 4, hook the old right subtree onto it, move
the left subtree over to `right`, set `left = None`, and carry on.
No list, no stack, no recursion.

**D - recursion** *(optional, after the others)*
Same leap of faith as always: assume `flatten(root.left)` and `flatten(root.right)`
already did their job perfectly. What is the one combine step left for you?
Careful - the recursive call returns `None` (it mutates in place), so you need a
way to reach the **end** of a flattened chain.

In [33]:
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right
    def preorderPrint(self):
        if self is None: return
        print(self.val, end=" ")
        if self.left: self.left.preorderPrint()
        if self.right: self.right.preorderPrint()



# Root
root = TreeNode(10)

# Level 1
root.left = TreeNode(5)
root.right = TreeNode(15)

# Level 2
root.left.left = TreeNode(3)
root.left.right = TreeNode(7)
root.right.left = TreeNode(12)
root.right.right = TreeNode(18)

# Level 3
root.left.left.left = TreeNode(1)
root.left.left.right = TreeNode(4)
root.right.left.right = TreeNode(13)
root.right.right.left = TreeNode(17)
root.right.right.right = TreeNode(20)
root.preorderPrint()

class Node:
    def __init__(self, fruit:TreeNode, next = None):
        self.fruit = fruit
        self.next = next

class Stack:
    def __init__(self , head:Node = None , last:Node = None ):
        self.head = head
        self.last = last
    def append(self,node:TreeNode):
        i = Node(node)
        if self.head is None :
            self.head = i
            self.last = i
        else:
            self.last.next = i
            self.last = i

    def preorder(self,root:TreeNode):
        if root is None: return
        self.append(root)
        if root.left: self.preorder(root.left)
        if root.right: self.preorder(root.right)


class Solution:

    def flatten(self, root: TreeNode) -> None:
        if root is None  : return

        stack = Stack()
        stack.preorder(root)
        if stack.head :
            current = stack.head.next
            currentroot = root
            while current:
                currentroot.left = None
                currentroot.right = current.fruit
                currentroot = currentroot.right
                current = current.next
            currentroot.left = None
            currentroot.right = None


Solution().flatten(root)
print("\n")
root.preorderPrint()




10 5 3 1 4 7 15 12 13 18 17 20 

10 5 3 1 4 7 15 12 13 18 17 20 

## Tests

`build` makes a tree from the LeetCode level-order list (with `None` for a
missing child). `chain` walks the result along `right` and **fails loudly if any
node still has a left child** - because "flattened" means both things: right
order *and* no left pointers.

In [34]:
def build(values: list) -> TreeNode:
    """LeetCode level-order list -> tree. Given to you, not part of the exercise."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q = deque([root])
    i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values):
            v = values[i]; i += 1
            if v is not None:
                node.left = TreeNode(v); q.append(node.left)
        if i < len(values):
            v = values[i]; i += 1
            if v is not None:
                node.right = TreeNode(v); q.append(node.right)
    return root


def chain(root: TreeNode) -> list:
    """Walk the flattened tree along .right and check every .left is None."""
    out = []
    cur = root
    while cur:
        if cur.left is not None:
            raise AssertionError(f"node {cur.val} still has a left child")
        out.append(cur.val)
        cur = cur.right
    return out


TESTS = [
    ([1, 2, 5, 3, 4, None, 6], [1, 2, 3, 4, 5, 6]),
    ([],                       []),
    ([0],                      [0]),
    ([1, 2],                   [1, 2]),
    ([1, None, 2],             [1, 2]),
    ([1, 2, 3, 4, 5, 6, 7],    [1, 2, 4, 5, 3, 6, 7]),
]

for values, expected in TESTS:
    root = build(values)
    Solution().flatten(root)
    got = chain(root)
    print(f"{str(values):26} -> {str(got):26} {'OK' if got == expected else 'FAIL, want ' + str(expected)}")

[1, 2, 5, 3, 4, None, 6]   -> [1, 2, 3, 4, 5, 6]         OK
[]                         -> []                         OK
[0]                        -> [0]                        OK
[1, 2]                     -> [1, 2]                     OK
[1, None, 2]               -> [1, 2]                     OK
[1, 2, 3, 4, 5, 6, 7]      -> [1, 2, 4, 5, 3, 6, 7]      OK


## After it passes

- **Is it really in-place?** Route A allocates a list of `n` nodes; route C
  allocates nothing. Prove it the way you proved `reverseList` was `O(1)`: check
  `id()` of a few nodes before and after and confirm you reused them instead of
  building new ones.
- **Which routes did you do, and what does each cost?** Write the time and space
  for A, B and C in one line each. If two of them are `O(n)` time, the difference
  is entirely in the space column - that is the whole point of the follow-up.
- Note `flatten` returns `None`. A function that works by **mutating its
  argument** is a different contract from one that **returns a new thing** - and
  it is why the tests check `root` after the call instead of a return value.